In [1]:
import os
import random
import numpy as np
import torch

# Change directory to the project root so that we can
# refer to in-house package as "src".
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import sqlite3
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# from sklearn.model_selection import train_test_split

# Clean package imports from the src/ directory
from src.data_loader import load_and_merge_data
from src.preprocessor import DataPreprocessor
from src.walmart_dataset import WalmartDataset
from src.sales_predictor import SalesPredictor

# Define a function to set the seed for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # If CUDA is available, make sure to set the seed for it too.
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Force deterministic C++ and CUDA libraries to execute deterministically.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 

set_seed(123)
print("Package imports and random seeds successfully initialized!")

Package imports and random seeds successfully initialized!


In [2]:
# Load the raw merged data frame.
df = load_and_merge_data('data')

# Sort the data chronologically to maintain time-series continuity.
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# Split train.csv chronologically to create a train and validation set.
train_df = df[df['Date'].dt.year < 2012]
val_df = df[df['Date'].dt.year == 2012]

# Pre-process and scale features
preprocessor = DataPreprocessor()

# Fit and transform only on data from 2010-2011
x_train, y_train = preprocessor.fit_transform(train_df)
# Only transform applied to 2012 validation data 
x_val, y_val = preprocessor.transform(val_df)

# Wrap the data in a PyTorch Dataset and Dataloader for batch training
train_set = WalmartDataset(x_train, y_train)
val_set = WalmartDataset(x_val, y_val)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False)

# Initialize the model
model = SalesPredictor(x_train.shape[1])

# Define the loss function and create the optimizer
criterion = nn.HuberLoss(delta=1.0) 
optimizer = optim.Adam(model.parameters(), lr=0.001) # For simplicity, using Adam as the optimizer.

In [3]:
# Training configurations.
epochs = 30
patience = 5 # Number of epochs to wait for early stopping.
best_val_loss = float('inf')
early_stopping_counter = 0

# Inference loop
for epoch in range(epochs):
    # Training phase.
    model.train()
    train_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Reset gradients.
        optimizer.zero_grad()

        # Forward pass through the model.
        predictions = model(inputs)

        # Compute the loss
        loss = criterion(predictions, targets)

        # Backward pass (to compute the gradients).
        loss.backward()

        # Optimizer step (to update the model parameters).
        optimizer.step()

        # Accumulate the batch loss
        train_loss += loss.item() * inputs.size(0)

    train_loss_scaled = train_loss / len(train_loader.dataset)
    # Real error. Apply an inverse transform to display real-world dollar
    # amount error (via preprocessor.y_scaler).
    real_train_error = train_loss_scaled * preprocessor.y_scaler.scale_[0]

    # Validation phase
    model.eval()
    val_loss = 0.0

    # Disable gradient computations for validation to save on memory and computations.
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            predictions = model(inputs)
            loss = criterion(predictions, targets)
            val_loss += loss.item() * inputs.size(0)

    val_loss_scaled = val_loss / len(val_loader.dataset)
    real_val_error = val_loss_scaled * preprocessor.y_scaler.scale_[0]

    print(f"Epoch {epoch+1}/epochs | Train loss: ${real_train_error:.2f} | Val. loss: ${real_val_error}")

    # Early stopping check
    if real_val_error < best_val_loss:
        best_val_loss = real_val_error
        early_stopping_counter = 0
        # Save the best weights
        torch.save(model.state_dict(), 'saved_weights/best_model.pth')
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print(f"\nEarly stopping now! Validation loss has not improved for {patience} epochs.")    

Epoch 1/epochs | Train loss: $5743.09 | Val. loss: $5426.35758779392
Epoch 2/epochs | Train loss: $5508.78 | Val. loss: $5260.639354089095
Epoch 3/epochs | Train loss: $5399.66 | Val. loss: $5113.441476826223
Epoch 4/epochs | Train loss: $5284.72 | Val. loss: $4906.975732948045
Epoch 5/epochs | Train loss: $5125.55 | Val. loss: $4482.519027132482
Epoch 6/epochs | Train loss: $4860.42 | Val. loss: $4209.731131251443
Epoch 7/epochs | Train loss: $4707.75 | Val. loss: $4211.999094881142
Epoch 8/epochs | Train loss: $4594.90 | Val. loss: $4136.298206245545
Epoch 9/epochs | Train loss: $4533.50 | Val. loss: $4270.077895942657
Epoch 10/epochs | Train loss: $4484.48 | Val. loss: $3981.531216122761
Epoch 11/epochs | Train loss: $4459.01 | Val. loss: $4134.762313584816
Epoch 12/epochs | Train loss: $4426.79 | Val. loss: $4104.547786800742
Epoch 13/epochs | Train loss: $4398.75 | Val. loss: $3905.007777323307
Epoch 14/epochs | Train loss: $4366.54 | Val. loss: $4314.723554259718
Epoch 15/epochs 

In [6]:
# Load the best model weights back in before testing inference.
model.load_state_dict(torch.load('saved_weights/best_model.pth', weights_only=True))
print("Best model weights restored. Onto testing...")

Best model weights restored. Onto testing...
